In [3]:
from huggingface_hub import notebook_login

notebook_login()

In [4]:
import torch
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback
from torch.optim import AdamW
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import evaluate

## LLama 2

In [35]:
model_llama = "meta-llama/Llama-2-7b-hf"

In [36]:
tokenizer = AutoTokenizer.from_pretrained(model_llama)
model = AutoModelForCausalLM.from_pretrained(
    model_llama,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [38]:
# Use model explicitly for inference clearly
prompt = "What is the highest mountain in the world"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_length=50, max_new_tokens=256, top_p=0.9, min_p=0.1, temperature=1.0, do_sample=True, no_repeat_ngram_size=2)

response = tokenizer.decode(output.squeeze(), skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is the highest mountain in the world?
Mount Everest, also known as Mount Chomolungma, is located in Nepal. The peak of Everst is 29,028 feet above sea level, making it the tallest mountain on Earth. It is also the center of the Everester.
The Everster is a group of climbers who have climbed Everset to its summit. To climb Eversets, you must be very careful. There are many risks that can be fatal if not taken seriously. Climbers need to know how to use the right gear, and they need good climbing skills. If you don’t have the skills or the gears, it is best not to go up the mountain. In 1924, George Mallory and Andrew Irvine attempted to reach the sumit. They were last seen on the South Col, about 8,850 meters (26,100 feet) above the sea, but their bodies were never found. No one has ever climed to the top since then. This is why Mount Everets is called the “dead mountain”.
What are the other tall mountains in world
K2 (8611m)



## OPT

In [32]:
model_opt = "facebook/opt-350m"

In [33]:
tokenizer = AutoTokenizer.from_pretrained(model_opt)
model = AutoModelForCausalLM.from_pretrained(
    model_opt,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
# Use model explicitly for inference clearly
prompt = "What is the highest mountain in the world?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_length=50, max_new_tokens=256, top_p=0.9, min_p=0.1, temperature=1.0, do_sample=True, no_repeat_ngram_size=2)

response = tokenizer.decode(output.squeeze(), skip_special_tokens=True)
print(response)

Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


What is the highest mountain in the world? I can't tell.
The highest mountain in the United States.


In [34]:
model.generate??

Signature:
model.generate(
    inputs: Optional[torch.Tensor] = None,
    generation_config: Optional[transformers.generation.configuration_utils.GenerationConfig] = None,
    logits_processor: Optional[transformers.generation.logits_process.LogitsProcessorList] = None,
    stopping_criteria: Optional[transformers.generation.stopping_criteria.StoppingCriteriaList] = None,
    prefix_allowed_tokens_fn: Optional[Callable[[int, torch.Tensor], List[int]]] = None,
    synced_gpus: Optional[bool] = None,
    assistant_model: Optional[ForwardRef('PreTrainedModel')] = None,
    streamer: Optional[ForwardRef('BaseStreamer')] = None,
    negative_prompt_ids: Optional[torch.Tensor] = None,
    negative_prompt_attention_mask: Optional[torch.Tensor] = None,
    **kwargs,
) -> Union[transformers.generation.utils.GenerateDecoderOnlyOutput, transformers.generation.utils.GenerateEncoderDecoderOutput, transformers.generation.utils.GenerateBeamDecoderOnlyOutput, transformers.generation.utils.GenerateBeam

## Dynamic MoE

In [16]:
model_dynamic = "AnLan577/Dynamic_MoE"

In [17]:
tokenizer = AutoTokenizer.from_pretrained(model_dynamic)
model = AutoModelForCausalLM.from_pretrained(
    model_dynamic,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at AnLan577/Dynamic_MoE and are newly initialized: ['model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.10.input_layernorm.weight', 'model.layers.10.mlp.down_proj.weight', 'model.layers.10.mlp.gate_proj.weight', 'model.layers.10.mlp.up_proj.weight', 'model.layers.10.post_attention_layernorm.weight', 'model.layers.11.input_layernorm.weight', 'model.layers.11.mlp.down_proj.weight', 'model.layers.11.mlp.gate_proj.weight', 'model.layers.11.mlp.up_proj.weight', 'model.layers.11.post_attention_layernorm.weight', 'model.layers.12.input_layernorm.w

In [20]:
# Use model explicitly for inference clearly
prompt = "The highest mountain in the world is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_length=100)

response = tokenizer.decode(output.squeeze(), skip_special_tokens=True)
print(response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The highest mountain in the world is


In [22]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="AnLan577/Dynamic_MoE", torch_dtype=torch.float16,
    device=0)
pipe("The highest mountain in the world is")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of LlamaForCausalLM were not initialized from the model checkpoint at AnLan577/Dynamic_MoE and are newly initialized: ['model.layers.0.input_layernorm.weight', 'model.layers.0.mlp.down_proj.weight', 'model.layers.0.mlp.gate_proj.weight', 'model.layers.0.mlp.up_proj.weight', 'model.layers.0.post_attention_layernorm.weight', 'model.layers.1.input_layernorm.weight', 'model.layers.1.mlp.down_proj.weight', 'model.layers.1.mlp.gate_proj.weight', 'model.layers.1.mlp.up_proj.weight', 'model.layers.1.post_attention_layernorm.weight', 'model.layers.10.input_layernorm.weight', 'model.layers.10.mlp.down_proj.weight', 'model.layers.10.mlp.gate_proj.weight', 'model.layers.10.mlp.up_proj.weight', 'model.layers.10.post_attention_layernorm.weight', 'model.layers.11.input_layernorm.weight', 'model.layers.11.mlp.down_proj.weight', 'model.layers.11.mlp.gate_proj.weight', 'model.layers.11.mlp.up_proj.weight', 'model.layers.11.post_attention_layernorm.weight', 'model.layers.12.input_layernorm.w

[{'generated_text': 'The highest mountain in the world isjer...,oser thro liv..., "$oser...,oserdz cz $.̯ $(" cz~~~~~~~~color Notification'}]